# Cleaning the enriched venue data

This notebook loads the enriched venue JSON file, flattens the nested accessibility information, cleans the data and saves the results.

The final files will be stored in the `data/cleaned` folder.

In [1]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\zasht\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Set the file paths

The notebook is stored inside `data/scripts`.

The input file is read from the `data/raw` folder, and the cleaned files will be saved inside `data/cleaned`.

In [3]:
from pathlib import Path
import json
import pandas as pd

input_path = Path("../raw/venues_enriched.json")
output_directory = Path("../cleaned")

output_directory.mkdir(parents=True, exist_ok=True)

## Load the enriched venue data

The enriched JSON file contains the venue information returned by Geoapify, along with the additional family-accessibility fields.

The file is loaded as a Python list of venue records.

In [4]:
with open(input_path, "r", encoding="utf-8") as file:
    venues = json.load(file)

print(f"{len(venues)} venues loaded")

400 venues loaded


## Flatten the nested JSON structure

The family-accessibility and verification information is currently nested inside each venue record.

`pd.json_normalize()` converts these nested values into separate DataFrame columns so they can be cleaned and imported into SQL.

In [5]:
df = pd.json_normalize(venues)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

display(df.head())

Rows: 400
Columns: 22


,place_id,name,area,town,type_of_place,address,postcode,latitude,longitude,website,...,family_accessibility.prams_allowed,family_accessibility.pram_storage,family_accessibility.changing_facilities,family_accessibility.additional_provisions,family_accessibility.table_reservation,family_accessibility.breastfeeding_friendly,family_accessibility.childrens_activities,family_accessibility.accessible_toilets,verification.source,verification.verified_at
0,513961c26856b6cabf5945d4e9e51ce74940f00103f901...,Humphreys,Hertfordshire,Welwyn Garden City,"[catering, catering.cafe]","Humphreys, The Campus, Welwyn Garden City, AL8...",AL8 6BX,51.805569,-0.208690,NaN,...,None,None,None,NaN,None,None,None,None,Geoapify/OpenStreetMap,None
1,51ec2a49433289cabf5912fa997adde64940f00103f901...,John Lewis Cafe,Hertfordshire,Welwyn Garden City,"[catering, catering.cafe]","John Lewis Cafe, Bridge Road, Welwyn Garden Ci...",AL8 6TP,51.803634,-0.207312,NaN,...,None,None,None,NaN,None,None,None,None,Geoapify/OpenStreetMap,None
2,51bd5296218e75cabf5946cc913aa6e64940f00103f901...,Caffè Nero,Hertfordshire,Welwyn Garden City,"[catering, catering.cafe, catering.cafe.coffee...","Caffè Nero, 31 Howardsgate, Welwyn Garden City...",AL8 6AP,51.801948,-0.206712,https://www.caffenero.com/uk/stores/welwyn-gar...,...,None,None,None,NaN,None,None,None,None,Geoapify/OpenStreetMap,None
3,515fb939f0c572cabf591690acb2a5e64940f00103f901...,Simmons,Hertfordshire,Welwyn Garden City,"[catering, catering.cafe]","Simmons, 33 Howardsgate, Welwyn Garden City, A...",AL8 6AP,51.801932,-0.206628,NaN,...,None,None,None,NaN,None,None,None,None,Geoapify/OpenStreetMap,None
4,51a4abeafad837cabf597708f340bae64940f00102f901...,Bebo,Hertfordshire,Welwyn Garden City,"[building, building.catering, catering, cateri...","Bebo, Stonehills, Welwyn Garden City, AL8 6NA,...",AL8 6NA,51.802559,-0.204829,NaN,...,None,None,None,NaN,None,None,None,None,Geoapify/OpenStreetMap,None


## Review the column names

The column names are reviewed before renaming them. Nested fields will currently contain a full stop, such as `family_accessibility.changing_facilities`.

In [6]:
df.columns.tolist()

['place_id',
 'name',
 'area',
 'town',
 'type_of_place',
 'address',
 'postcode',
 'latitude',
 'longitude',
 'website',
 'opening_hours',
 'family_accessibility.accessible_entrance',
 'family_accessibility.prams_allowed',
 'family_accessibility.pram_storage',
 'family_accessibility.changing_facilities',
 'family_accessibility.additional_provisions',
 'family_accessibility.table_reservation',
 'family_accessibility.breastfeeding_friendly',
 'family_accessibility.childrens_activities',
 'family_accessibility.accessible_toilets',
 'verification.source',
 'verification.verified_at']

## Rename the flattened columns

The nested column names are shortened and standardised. This makes them easier to reference in Python and aligns them with the planned SQL database fields.

In [7]:
df = df.rename(columns={
    "family_accessibility.accessible_entrance": "accessible_entrance",
    "family_accessibility.prams_allowed": "prams_allowed",
    "family_accessibility.pram_storage": "pram_storage",
    "family_accessibility.changing_facilities": "changing_facilities",
    "family_accessibility.additional_provisions": "additional_provisions",
    "family_accessibility.table_reservation": "table_reservation",
    "family_accessibility.breastfeeding_friendly": "breastfeeding_friendly",
    "family_accessibility.childrens_activities": "childrens_activities",
    "family_accessibility.accessible_toilets": "accessible_toilet",
    "verification.source": "accessibility_source",
    "verification.verified_at": "accessibility_verified_at"
})

## Confirm the renamed columns

The updated column list is displayed to check that the nested accessibility and verification fields were renamed successfully.

In [8]:
df.columns.tolist()

['place_id',
 'name',
 'area',
 'town',
 'type_of_place',
 'address',
 'postcode',
 'latitude',
 'longitude',
 'website',
 'opening_hours',
 'accessible_entrance',
 'prams_allowed',
 'pram_storage',
 'changing_facilities',
 'additional_provisions',
 'table_reservation',
 'breastfeeding_friendly',
 'childrens_activities',
 'accessible_toilet',
 'accessibility_source',
 'accessibility_verified_at']

## Check for duplicate venues

The same venue may have been returned under more than one Geoapify category.

The `place_id` uniquely identifies each venue, so it is used to count duplicate records.

In [9]:
duplicate_count = df.duplicated(subset="place_id").sum()

print(f"Duplicate venues found: {duplicate_count}")

Duplicate venues found: 0


## Remove duplicate venues

Duplicate records are removed using `place_id`.

The first occurrence of each venue is retained.

In [10]:
before = len(df)

df = df.drop_duplicates(
    subset="place_id",
    keep="first"
).copy()

after = len(df)

print(f"Venues before cleaning: {before}")
print(f"Duplicates removed: {before - after}")
print(f"Venues remaining: {after}")

Venues before cleaning: 400
Duplicates removed: 0
Venues remaining: 400


## Clean the text columns

Leading and trailing spaces are removed from the main text fields.

Empty strings are replaced with missing values so that they are treated consistently as unknown data.

In [11]:
text_columns = [
    "name",
    "area",
    "town",
    "address",
    "postcode",
    "website",
    "opening_hours",
    "additional_provisions"
]

for column in text_columns:
    if column in df.columns:
        df[column] = df[column].apply(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

        df[column] = df[column].replace("", pd.NA)

## Standardise postcodes

Whitespace is removed from the beginning and end of each postcode, and letters are converted to uppercase.

Missing postcodes remain empty rather than being replaced with inaccurate information.

In [12]:
df["postcode"] = df["postcode"].apply(
    lambda value: value.strip().upper()
    if isinstance(value, str)
    else value
)

## Review missing values

This displays the number of missing values in each column.

Missing accessibility values will remain unknown and will not automatically be changed to `False`.

In [13]:
missing_values = df.isna().sum().sort_values(ascending=False)

display(missing_values)

breastfeeding_friendly       400
childrens_activities         400
accessibility_verified_at    400
table_reservation            400
pram_storage                 400
prams_allowed                400
changing_facilities          400
additional_provisions        399
accessible_toilet            396
accessible_entrance          379
opening_hours                304
website                      266
area                          41
postcode                       1
address                        0
type_of_place                  0
town                           0
name                           0
place_id                       0
longitude                      0
latitude                       0
accessibility_source           0
dtype: int64

## Standardise accessibility values

Accessibility fields should contain:

- `True` when an amenity is confirmed.
- `False` when an amenity is confirmed as unavailable.
- A missing value when the information is unknown.

The value `limited` is kept as unknown for the Boolean SQL fields because it is neither fully true nor false.

In [15]:
accessibility_columns = [
    "accessible_entrance",
    "accessible_toilet",
    "prams_allowed",
    "pram_storage",
    "changing_facilities",
    "table_reservation",
    "breastfeeding_friendly",
    "childrens_activities"
]

for column in accessibility_columns:
    if column in df.columns:
        df[column] = df[column].replace({
            "yes": True,
            "no": False,
            "true": True,
            "false": False,
            "limited": pd.NA
        })

## Simplify the venue category

Geoapify can return several categories for one venue. For this MVP, the first category is selected as the venue's main category.

The column is renamed from `type_of_place` to `category` to match the SQL table.

In [16]:
df["category"] = df["type_of_place"].apply(
    lambda categories: categories[0]
    if isinstance(categories, list) and len(categories) > 0
    else pd.NA
)

## Rename fields to match the SQL database

The Geoapify place identifier is renamed so that it matches the planned `venues` SQL table.

In [17]:
df = df.rename(columns={
    "place_id": "geoapify_place_id",
    "area": "county"
})

## Select and order the final columns

Only the fields required by the application and SQL database are retained.

Database-generated fields such as `id` and `created_at` are not needed in the JSON file.

In [18]:
final_columns = [
    "geoapify_place_id",
    "name",
    "category",
    "address",
    "town",
    "county",
    "postcode",
    "latitude",
    "longitude",
    "website",
    "opening_hours",
    "accessible_entrance",
    "accessible_toilet",
    "prams_allowed",
    "pram_storage",
    "changing_facilities",
    "table_reservation",
    "breastfeeding_friendly",
    "childrens_activities",
    "additional_provisions",
    "accessibility_source",
    "accessibility_verified_at"
]

df_cleaned = df[final_columns].copy()

print(df_cleaned.shape)
display(df_cleaned.head())

(400, 22)


,geoapify_place_id,name,category,address,town,county,postcode,latitude,longitude,website,...,accessible_toilet,prams_allowed,pram_storage,changing_facilities,table_reservation,breastfeeding_friendly,childrens_activities,additional_provisions,accessibility_source,accessibility_verified_at
0,513961c26856b6cabf5945d4e9e51ce74940f00103f901...,Humphreys,catering,"Humphreys, The Campus, Welwyn Garden City, AL8...",Welwyn Garden City,Hertfordshire,AL8 6BX,51.805569,-0.208690,NaN,...,None,None,None,None,None,None,None,NaN,Geoapify/OpenStreetMap,None
1,51ec2a49433289cabf5912fa997adde64940f00103f901...,John Lewis Cafe,catering,"John Lewis Cafe, Bridge Road, Welwyn Garden Ci...",Welwyn Garden City,Hertfordshire,AL8 6TP,51.803634,-0.207312,NaN,...,None,None,None,None,None,None,None,NaN,Geoapify/OpenStreetMap,None
2,51bd5296218e75cabf5946cc913aa6e64940f00103f901...,Caffè Nero,catering,"Caffè Nero, 31 Howardsgate, Welwyn Garden City...",Welwyn Garden City,Hertfordshire,AL8 6AP,51.801948,-0.206712,https://www.caffenero.com/uk/stores/welwyn-gar...,...,None,None,None,None,None,None,None,NaN,Geoapify/OpenStreetMap,None
3,515fb939f0c572cabf591690acb2a5e64940f00103f901...,Simmons,catering,"Simmons, 33 Howardsgate, Welwyn Garden City, A...",Welwyn Garden City,Hertfordshire,AL8 6AP,51.801932,-0.206628,NaN,...,None,None,None,None,None,None,None,NaN,Geoapify/OpenStreetMap,None
4,51a4abeafad837cabf597708f340bae64940f00102f901...,Bebo,building,"Bebo, Stonehills, Welwyn Garden City, AL8 6NA,...",Welwyn Garden City,Hertfordshire,AL8 6NA,51.802559,-0.204829,NaN,...,None,None,None,None,None,None,None,NaN,Geoapify/OpenStreetMap,None


## Save the cleaned venue data

The cleaned data is saved as:

- JSON for importing into PostgreSQL.
- CSV for reviewing the records in Excel.

Missing values are exported as `null` in the JSON file.

In [19]:
json_output_path = output_directory / "venues_cleaned.json"
csv_output_path = output_directory / "venues_cleaned.csv"

df_cleaned.to_json(
    json_output_path,
    orient="records",
    indent=2,
    force_ascii=False
)

df_cleaned.to_csv(
    csv_output_path,
    index=False
)

print(f"JSON saved to: {json_output_path}")
print(f"CSV saved to: {csv_output_path}")
print(f"{len(df_cleaned)} cleaned venues saved")

JSON saved to: ..\cleaned\venues_cleaned.json
CSV saved to: ..\cleaned\venues_cleaned.csv
400 cleaned venues saved
